# Пример - Полиформы орграфов

## Подготовка

Выполните все ячейки сверху вниз. Ревизия пакета: **f84dc6a**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/f84dc6abefcc6adaea313f97395eeb0587b337ea.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


С одной стрелкой связаны два разных расчёта. При увеличении веса $a\to b$ на единицу остовные потенциалы меняются на $(0,2,1)$. Измерение этого направления даёт число $1/4$. Выполним оба расчёта на одном графе, а затем по таблице резистенсов восстановим дуги и измерим границу.


Выполняйте ячейки сверху вниз. Блокнот устанавливает закреплённую ревизию пакета. [polyform_algebra f84dc6a](https://github.com/magin67/polyform_algebra/commit/f84dc6abefcc6adaea313f97395eeb0587b337ea).


## 1. От дуг к полиформе

Зададим шесть дуг. Пара $(b,a)$ означает дугу из $b$ в $a$ и меняет строку $b$ лапласиана. У $b$ исходящий вес равен 3, входящий равен 2.



В порядке $a,b,c$ выпишем $M=\exp L$ по грейдам. В матрице второго грейда строки и столбцы идут в порядке $a\wedge b,a\wedge c,b\wedge c$. Граница $(abc)$ имеет в этом базисе координаты $(1,-1,1)$.

[Как из дуг получается полиформа орграфа](https://magin67.github.io/pmg-notes/ru/%D0%BD%D0%B0%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D1%8B/%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BA%D0%B0%D0%BA-%D0%B8%D0%B7-%D0%B4%D1%83%D0%B3-%D0%BF%D0%BE%D0%BB%D1%83%D1%87%D0%B0%D0%B5%D1%82%D1%81%D1%8F-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D0%B0-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%B0) · [Как вычислить остовной потенциал вершины по полиформе орграфа](https://magin67.github.io/pmg-notes/ru/%D0%BD%D0%B0%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D1%8B/%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BA%D0%B0%D0%BA-%D0%B2%D1%8B%D1%87%D0%B8%D1%81%D0%BB%D0%B8%D1%82%D1%8C-%D0%BE%D1%81%D1%82%D0%BE%D0%B2%D0%BD%D0%BE%D0%B9-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D1%8B-%D0%BF%D0%BE-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D0%B5-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%B0)

In [3]:
from itertools import combinations
from IPython.display import Math, display
from sympy import Matrix, Rational, eye, latex, ones, symbols, zeros
from polyform_algebra.core import DomainError
from polyform_algebra.directed import DirectedNetwork

a, b, c, d, e = tuple("abcde")
vertices = (a, b, c)
arcs = ((a, b, 1), (b, a, 2), (a, c, 2), (c, a, 1), (b, c, 1), (c, b, 1))
graph = DirectedNetwork(vertices, arcs)
L = graph.laplacian()
M2 = graph.polyform_grade(2)
boundary = Matrix([[1, -1, 1]])
assert graph.polyform_grade(0) == eye(1)
assert graph.polyform_grade(1) == L
assert graph.polyform_grade(3) == zeros(1)
assert M2 == M2[:, 0] * boundary
display(Math("L=" + latex(L) + r",\qquad M_2=" + latex(M2)))
# Present the left blade in cyclic order bc, ca, ab.
bc, ca, ab = M2[2, 0], -M2[1, 0], M2[0, 0]
display(Math(r"M=e+L+[" + rf"{bc}\,b\wedge c+{ca}\,c\wedge a+{ab}\,a\wedge b"
             + r",\,(abc)]"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [4]:
# On three vertices two arcs form a tree if their tails differ and they do not form a cycle.
tree_weights = {v: [] for v in vertices}
for (i, j, w), (k, m, z) in combinations(arcs, 2):
    if i == k or (i, j) == (m, k):
        continue
    root, = set(vertices) - {i, k}
    tree_weights[root].append(w * z)
tau = graph.tree_potentials()
assert Matrix(tau).T * L == zeros(1, 3)
assert tuple(map(sum, tree_weights.values())) == (bc, ca, ab) == tau == (5, 4, 7)
for root in vertices:
    print(root, sorted(tree_weights[root]), "→", sum(tree_weights[root]))

a [1, 2, 2] → 5
b [1, 1, 2] → 4
c [1, 2, 4] → 7


## 2. Усилить дугу или измерить пару

У формы дуги $H_{a\to b}=[a,a-b]$ слева стоит вершина $a$. У измерительной формы $\Phi_{ab}=[b-h,b-a]$ слева стоит вектор из центроида всех вершин в $b$. Методы возвращают два координатных столбца $(x,y)$ в порядке вершин графа. Матрица формы равна $xy^{\mathsf T}$.

Сначала прибавим $t$ к весу дуги. Для каждого корня вычислим главный минор возмущённого лапласиана и отдельно сверим его коэффициент при $t$ с результатом пакета.

[Форма дуги и форма одностороннего резистенса](https://magin67.github.io/pmg-notes/ru/%D0%BD%D0%B0%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D1%8B/%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%84%D0%BE%D1%80%D0%BC%D0%B0-%D0%B4%D1%83%D0%B3%D0%B8-%D0%B8-%D1%84%D0%BE%D1%80%D0%BC%D0%B0-%D0%BE%D0%B4%D0%BD%D0%BE%D1%81%D1%82%D0%BE%D1%80%D0%BE%D0%BD%D0%BD%D0%B5%D0%B3%D0%BE-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%B0) · [Как нормировать односторонний резистенс в небалансовом орграфе](https://magin67.github.io/pmg-notes/ru/%D0%BD%D0%B0%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D1%8B/%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BA%D0%B0%D0%BA-%D0%BD%D0%BE%D1%80%D0%BC%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D1%82%D1%8C-%D0%BE%D0%B4%D0%BD%D0%BE%D1%81%D1%82%D0%BE%D1%80%D0%BE%D0%BD%D0%BD%D0%B8%D0%B9-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81-%D0%B2-%D0%BD%D0%B5%D0%B1%D0%B0%D0%BB%D0%B0%D0%BD%D1%81%D0%BE%D0%B2%D0%BE%D0%BC-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%B5)

In [5]:
t = symbols("t")
x, y = graph.arc_form(a, b)
changed = L + t * x * y.T
root_polynomials = tuple(changed.minor_submatrix(i, i).det().expand() for i in range(3))
increments = graph.root_coefficients(x, y)
assert increments == (0, 2, 1)
assert root_polynomials == tuple(old + t * inc for old, inc in zip(tau, increments))
display(Math(r"(\tau_a(t),\tau_b(t),\tau_c(t))=" + latex(root_polynomials)))

<IPython.core.display.Math object>

In [6]:
# Keep the same graph and evaluate the two ordered pairs.
for start, target in [(a, b), (b, a)]:
    form = graph.measurement_form(start, target)
    u = graph.root_coefficients(*form)
    r = graph.normalized_potential(*form)
    assert r == sum(u) / sum(tau)
    print(f"{start} → {target}   u = {u}   r = {r}")

resistances = graph.one_way_resistances()
_, ab_vector = graph.measurement_form(a, b)
assert (resistances[0, 1], resistances[1, 0]) == (Rational(1, 4), Rational(3, 16))
pair = graph.normalized_potential(ab_vector, ab_vector)
assert resistances[0, 1] + resistances[1, 0] == pair == Rational(7, 16)
display(Math(r"r_{ab}+r_{ba}=" + latex(pair)))

a → b   u = (1, 4/3, 5/3)   r = 1/4
b → a   u = (1, 2/3, 4/3)   r = 3/16


<IPython.core.display.Math object>

## 3. Какие корни включать в сумму

В единичной цепочке $a\to b\to c\to d$ единственное остовное дерево ведёт к $d$. Выведем вклады **всех** корней при измерении двух внутренних направлений и двух направлений между концами. Для $(bc)$ вклады $b$ и $d$ сократятся.

У развилки $b\leftarrow a\to c$ два стока. Её полиформу и кофакторы можно вычислить, но $S=0$, поэтому численное измерение пары по принятому правилу не определено.

[Как вычислить остовной потенциал вершины по полиформе орграфа](https://magin67.github.io/pmg-notes/ru/%D0%BD%D0%B0%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D1%8B/%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BA%D0%B0%D0%BA-%D0%B2%D1%8B%D1%87%D0%B8%D1%81%D0%BB%D0%B8%D1%82%D1%8C-%D0%BE%D1%81%D1%82%D0%BE%D0%B2%D0%BD%D0%BE%D0%B9-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D1%8B-%D0%BF%D0%BE-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D0%B5-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%B0) · [Как нормировать односторонний резистенс в небалансовом орграфе](https://magin67.github.io/pmg-notes/ru/%D0%BD%D0%B0%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D1%8B/%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BA%D0%B0%D0%BA-%D0%BD%D0%BE%D1%80%D0%BC%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D1%82%D1%8C-%D0%BE%D0%B4%D0%BD%D0%BE%D1%81%D1%82%D0%BE%D1%80%D0%BE%D0%BD%D0%BD%D0%B8%D0%B9-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81-%D0%B2-%D0%BD%D0%B5%D0%B1%D0%B0%D0%BB%D0%B0%D0%BD%D1%81%D0%BE%D0%B2%D0%BE%D0%BC-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%B5)

In [7]:
def unit_path(labels, closed=False):
    targets = labels[1:] + (labels[:1] if closed else ())
    return DirectedNetwork(labels, [(i, j, 1) for i, j in zip(labels, targets)])

chain = unit_path((a, b, c, d))
assert chain.tree_potentials() == (0, 0, 0, 1)
for i, j, expected in [(b, c, 0), (c, b, 1), (a, d, 3), (d, a, 1)]:
    form = chain.measurement_form(i, j)
    u = chain.root_coefficients(*form)
    value = chain.normalized_potential(*form)
    assert value == expected
    print(f"{i} → {j}   u = {u}   r = {value}")
fork = DirectedNetwork(vertices, ((a, b, 1), (a, c, 1)))
assert fork.tree_potentials() == (0, 0, 0)
try:
    fork.one_way_resistances()
except DomainError:
    print("S = 0  →  нормировка не определена")
else:
    raise AssertionError("S = 0")

b → c   u = (0, -1/4, 0, 1/4)   r = 0
c → b   u = (0, 1/4, 0, 3/4)   r = 1
a → d   u = (3/4, 3/4, 3/4, 3/4)   r = 3
d → a   u = (1/4, 1/4, 1/4, 1/4)   r = 1
S = 0  →  нормировка не определена


## 4. Восстановить граф и вычислить НСП

Из таблицы $\mathbf r$ находим $\mathbf K=(-\mathbf J\mathbf r)^+$, где $\mathbf J$ вычитает среднее. Так восстанавливаются разности строк лапласиана. Доли $\pi_i=\tau_i/S$ позволяют вернуть и общий член строк. Проверим, что по одним и тем же числам таблицы на **двух** вершинах можно восстановить две противоположные дуги, если поменять местами доли корней.

Для направленного скалярного произведения достаточно самой таблицы. Первый аргумент $u$ ставится справа в формуле $-\mathbf v^{\mathsf T}\mathbf r\mathbf u$.

[Можно ли восстановить орграф по матрице односторонних резистенсов?](https://magin67.github.io/pmg-notes/ru/%D0%BD%D0%B0%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D1%8B/%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BC%D0%BE%D0%B6%D0%BD%D0%BE-%D0%BB%D0%B8-%D0%B2%D0%BE%D1%81%D1%81%D1%82%D0%B0%D0%BD%D0%BE%D0%B2%D0%B8%D1%82%D1%8C-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84-%D0%BF%D0%BE-%D0%BC%D0%B0%D1%82%D1%80%D0%B8%D1%86%D0%B5-%D0%BE%D0%B4%D0%BD%D0%BE%D1%81%D1%82%D0%BE%D1%80%D0%BE%D0%BD%D0%BD%D0%B8%D1%85-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%BE%D0%B2) · [Как вычислить направленное скалярное произведение двух векторов в орграфе?](https://magin67.github.io/pmg-notes/ru/%D0%BD%D0%B0%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D1%8B/%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BA%D0%B0%D0%BA-%D0%B2%D1%8B%D1%87%D0%B8%D1%81%D0%BB%D0%B8%D1%82%D1%8C-%D0%BD%D0%B0%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%BD%D0%BE%D0%B5-%D1%81%D0%BA%D0%B0%D0%BB%D1%8F%D1%80%D0%BD%D0%BE%D0%B5-%D0%BF%D1%80%D0%BE%D0%B8%D0%B7%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B4%D0%B2%D1%83%D1%85-%D0%B2%D0%B5%D0%BA%D1%82%D0%BE%D1%80%D0%BE%D0%B2-%D0%B2-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%B5)

In [8]:
J = eye(3) - ones(3) / 3
K = (-J * resistances).pinv()
pi = Matrix(tau) / sum(tau)
restored = DirectedNetwork.from_one_way_resistances(vertices, resistances, pi)
assert K == J * L
assert K - ones(3, 1) * pi.T * K == restored.laplacian() == L
assert restored.arcs == graph.arcs
display(Math(r"\mathbf r=" + latex(resistances) + r",\qquad\mathbf K=" + latex(K)))
display(Math(r"\mathbf L_{\rm recovered}=" + latex(restored.laplacian())))
same_table = [[0, 1], [1, 0]]
for shares in [(0, 1), (1, 0)]:
    candidate = DirectedNetwork.from_one_way_resistances((a, b), same_table, shares)
    assert candidate.one_way_resistances() == Matrix(same_table)
    print(shares, "→", candidate.arcs)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

(0, 1) → (('a', 'b', Fraction(1, 1)),)
(1, 0) → (('b', 'a', Fraction(1, 1)),)


In [9]:
_, u = graph.measurement_form(a, b)
_, v = graph.measurement_form(a, c)
for left, right, expected in [(u, v, Rational(1, 4)), (v, u, Rational(1, 8))]:
    value = graph.directed_scalar_product(left, right)
    assert value == -(right.T * resistances * left)[0] == graph.normalized_potential(left, right)
    assert value == expected
    print(value)

1/4
1/8


## 5. Измерить границу по началам и по грамиану

В 4-орцикле измерим треугольник $(abc)$, затем тетраэдр $(abcd)$. После этого вставим внешнюю вершину $e$ между $d$ и $a$ и повторим расчёт тетраэдра. Передаваемый список вершин задаёт границу; центроид по-прежнему учитывает весь орграф.



Для каждого случая пакет вычисляет вклады по начальным вершинам. Полную норму независимо проверяем по грамиану, собранному напрямую из таблицы резистенсов. Последний опыт обращает ориентацию треугольника при фиксированном начале.

[Что измеряет произведение форм двух сторон треугольника?](https://magin67.github.io/pmg-notes/ru/%D0%BD%D0%B0%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D1%8B/%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%87%D1%82%D0%BE-%D0%B8%D0%B7%D0%BC%D0%B5%D1%80%D1%8F%D0%B5%D1%82-%D0%BF%D1%80%D0%BE%D0%B8%D0%B7%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0%B8%D0%B5-%D1%84%D0%BE%D1%80%D0%BC-%D0%B4%D0%B2%D1%83%D1%85-%D1%81%D1%82%D0%BE%D1%80%D0%BE%D0%BD-%D1%82%D1%80%D0%B5%D1%83%D0%B3%D0%BE%D0%BB%D1%8C%D0%BD%D0%B8%D0%BA%D0%B0) · [Как вычислить норму границы произвольного порядка в орграфе?](https://magin67.github.io/pmg-notes/ru/%D0%BD%D0%B0%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D1%8B/%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BA%D0%B0%D0%BA-%D0%B2%D1%8B%D1%87%D0%B8%D1%81%D0%BB%D0%B8%D1%82%D1%8C-%D0%BD%D0%BE%D1%80%D0%BC%D1%83-%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%86%D1%8B-%D0%BF%D1%80%D0%BE%D0%B8%D0%B7%D0%B2%D0%BE%D0%BB%D1%8C%D0%BD%D0%BE%D0%B3%D0%BE-%D0%BF%D0%BE%D1%80%D1%8F%D0%B4%D0%BA%D0%B0-%D0%B2-%D0%BE%D1%80%D0%B3%D1%80%D0%B0%D1%84%D0%B5)

In [10]:
square = unit_path((a, b, c, d), closed=True)
pentagon = unit_path((a, b, c, d, e), closed=True)
for net, boundary_labels, numerators in [
    (square, (a, b, c), (1, 1, 2)),
    (square, (a, b, c, d), (1, 1, 1, 1)),
    (pentagon, (a, b, c, d), (1, 1, 1, 2)),
]:
    start = boundary_labels[0]
    directions = Matrix.hstack(*(net.measurement_form(start, j)[1] for j in boundary_labels[1:]))
    gram = -directions.T * net.one_way_resistances().T * directions
    parts = net.boundary_contributions(boundary_labels)
    assert list(parts.values()) == [Rational(x, len(net.labels)) for x in numerators]
    assert sum(parts.values()) == net.boundary_norm(boundary_labels) == gram.det() == 1
    print(f"{''.join(net.labels)} / {''.join(boundary_labels)}   {parts}   det = {gram.det()}")
assert square.boundary_contributions((a, b, c))[a] == square.boundary_contributions((a, c, b))[a]

abcd / abc   {'a': 1/4, 'b': 1/4, 'c': 1/2}   det = 1
abcd / abcd   {'a': 1/4, 'b': 1/4, 'c': 1/4, 'd': 1/4}   det = 1
abcde / abcd   {'a': 1/5, 'b': 1/5, 'c': 1/5, 'd': 2/5}   det = 1


Проверим вклад начала $c$ напрямую. Каждая поправка ранга один входит в минор линейно, поэтому четыре значения ниже выделяют ровно смешанный коэффициент.

In [11]:
# Independent coefficient of t1*t2 from principal minors, for start c of (cab).
def root_total(matrix):
    return sum(matrix.minor_submatrix(i, i).det() for i in range(matrix.rows))

x1, y1 = square.measurement_form(c, a)
x2, y2 = square.measurement_form(a, b)
A = square.laplacian()
F1, F2 = x1 * y1.T, x2 * y2.T
mixed = root_total(A + F1 + F2) - root_total(A + F1) - root_total(A + F2) + root_total(A)
direct = mixed / root_total(A)
assert direct == square.boundary_contributions((c, a, b))[c] == Rational(1, 2)
display(Math(r"P_M^\Sigma(\Phi_{ca}\Phi_{ab})=" + latex(direct)))

<IPython.core.display.Math object>

Перенос начала треугольника из $a$ в $c$ меняет вклад с $1/4$ на $1/2$. При обращении ориентации с сохранением начала $a$ вклад остаётся $1/4$. Вставка $e$ меняет доли для тетраэдра, а его полная норма остаётся равной 1.